# Healthcare claims quality

This notebook reads the committed healthcare claims CSV using the exact dataset schema,
applies claim quality rules, quarantines known anomalies, and publishes daily gold summaries.


In [ ]:
ENVIRONMENT = "dev"
PROCESSING_DATE = "2026-02-07"
PIPELINE_RUN_ID = "local-demo"
environment = ENVIRONMENT
processing_date = PROCESSING_DATE
pipeline_run_id = PIPELINE_RUN_ID


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T, Window
import json
import uuid

batch_id = f"claims-{processing_date}-{uuid.uuid4()}"

source_path = 'Files/landing/healthcare-claims/claims.csv'
bronze_table = 'bronze.healthcare_claims_raw'
silver_table = 'silver.healthcare_claims_validated'
quarantine_table = 'silver.healthcare_claims_quarantine'
gold_table = 'gold.claims_quality_daily'


In [ ]:
def log_event(stage, message, **metrics):
    record = {
        'stage': stage,
        'message': message,
        'environment': environment,
        'processing_date': processing_date,
        'pipeline_run_id': pipeline_run_id,
        'batch_id': batch_id,
        'metrics': metrics,
    }
    print(json.dumps(record, sort_keys=True))

claim_schema = T.StructType([
    T.StructField('claim_id', T.StringType(), False),
    T.StructField('member_id', T.StringType(), False),
    T.StructField('claim_received_date', T.DateType(), False),
    T.StructField('service_date', T.DateType(), False),
    T.StructField('provider_code', T.StringType(), False),
    T.StructField('procedure_code', T.StringType(), False),
    T.StructField('place_of_service', T.StringType(), False),
    T.StructField('claim_amount', T.DecimalType(18, 2), False),
    T.StructField('allowed_amount', T.DecimalType(18, 2), False),
    T.StructField('claim_status', T.StringType(), False),
    T.StructField('synthetic_record', T.StringType(), False),
    T.StructField('anomaly_label', T.StringType(), False),
])

for schema_name in ['bronze', 'silver', 'gold']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_name}')

raw_df = (
    spark.read.format('csv')
    .option('header', True)
    .schema(claim_schema)
    .load(source_path)
    .withColumn('metadata_ingestion_timestamp', F.current_timestamp())
    .withColumn('source_file', F.input_file_name())
    .withColumn('metadata_batch_id', F.lit(batch_id))
    .withColumn('metadata_pipeline_run_id', F.lit(pipeline_run_id))
    .withColumn('metadata_processing_date', F.lit(processing_date))
    .withColumn('claim_received_day', F.col('claim_received_date'))
)

raw_df.write.format('delta').mode('overwrite').partitionBy('claim_received_day').option('overwriteSchema', 'true').saveAsTable(bronze_table)


In [ ]:
cutoff_date = F.to_date(F.lit(processing_date))
claims_df = spark.table(bronze_table).filter(F.col('claim_received_date') <= cutoff_date)
claim_window = Window.partitionBy('claim_id').orderBy(F.col('claim_received_date').desc_nulls_last(), F.col('metadata_ingestion_timestamp').desc_nulls_last())

latest_df = (
    claims_df
    .withColumn('claim_rank', F.row_number().over(claim_window))
    .filter(F.col('claim_rank') == 1)
    .drop('claim_rank')
    .withColumn('service_year_month', F.date_format('service_date', 'yyyy-MM'))
)

quarantine_condition = (
    (F.col('anomaly_label') != F.lit('none'))
    | F.col('provider_code').isNull()
    | (F.col('provider_code') == '')
    | (F.col('claim_amount') < 0)
    | (F.col('allowed_amount') < 0)
    | (F.col('allowed_amount') > F.col('claim_amount'))
    | (F.col('service_date') > F.col('claim_received_date'))
)

quarantine_df = (
    latest_df
    .filter(quarantine_condition)
    .withColumn(
        'quarantine_reason',
        F.concat_ws(
            '; ',
            F.when(F.col('anomaly_label') != 'none', F.col('anomaly_label')),
            F.when(F.col('provider_code').isNull() | (F.col('provider_code') == ''), F.lit('missing_provider_code')),
            F.when(F.col('claim_amount') < 0, F.lit('invalid_claim_amount')),
            F.when(F.col('allowed_amount') < 0, F.lit('invalid_allowed_amount')),
            F.when(F.col('allowed_amount') > F.col('claim_amount'), F.lit('allowed_exceeds_claim')),
            F.when(F.col('service_date') > F.col('claim_received_date'), F.lit('service_after_received')),
        ),
    )
    .withColumn('quarantine_timestamp', F.current_timestamp())
)

valid_df = (
    latest_df
    .filter(~quarantine_condition)
    .withColumn('claim_variance', F.round(F.col('claim_amount') - F.col('allowed_amount'), 2))
    .withColumn('allowed_ratio', F.round(F.when(F.col('claim_amount') != 0, F.col('allowed_amount') / F.col('claim_amount')), 6))
)

if spark.catalog.tableExists(silver_table):
    target = DeltaTable.forName(spark, silver_table)
    (
        target.alias('target')
        .merge(valid_df.alias('source'), 'target.claim_id = source.claim_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    valid_df.write.format('delta').mode('overwrite').partitionBy('service_year_month').option('overwriteSchema', 'true').saveAsTable(silver_table)

quarantine_df.write.format('delta').mode('overwrite').partitionBy('service_year_month').option('overwriteSchema', 'true').saveAsTable(quarantine_table)


In [ ]:
quality_summary_df = (
    latest_df
    .groupBy('claim_received_date', 'claim_status', 'place_of_service')
    .agg(
        F.count('*').alias('claim_count'),
        F.sum(F.when(quarantine_condition, F.lit(1)).otherwise(F.lit(0))).alias('failed_rule_count'),
        F.round(F.sum('claim_amount'), 2).alias('total_claim_amount'),
        F.round(F.sum('allowed_amount'), 2).alias('total_allowed_amount'),
    )
    .withColumn('pipeline_run_id', F.lit(pipeline_run_id))
    .withColumn('quality_pass_rate', F.round(1 - (F.col('failed_rule_count') / F.col('claim_count')), 6))
    .withColumnRenamed('claim_received_date', 'processing_day')
)

quality_summary_df.write.format('delta').mode('overwrite').partitionBy('processing_day').option('overwriteSchema', 'true').saveAsTable(gold_table)

log_event(
    'claims_validation',
    'Healthcare claims processing complete',
    bronze_rows=raw_df.count(),
    silver_rows=valid_df.count(),
    quarantine_rows=quarantine_df.count(),
    gold_rows=quality_summary_df.count(),
)
spark.sql(f'SELECT COUNT(*) AS claims_gold_rows FROM {gold_table}').show()
